# 📈 Retail Sales Time Series Forecasting Engine
### Built by Nikhil Tandon
This notebook demonstrates an end-to-end time series analysis and forecasting pipeline using classical statistical models (SARIMAX) and advanced additive frameworks (Facebook Prophet).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
import warnings
warnings.filterwarnings('ignore')

## 1. Data Ingestion & Exploration

In [ ]:
# Load historical data file
df = pd.read_csv('time_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
print(f"Data Shape: {df.shape}")
df.head()

In [ ]:
# Visualize baseline sales trend
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['Sales'], marker='o', color='#7c3aed', linewidth=2)
plt.title('Historical Monthly Sales Volume Trajectory')
plt.xlabel('Timeline')
plt.ylabel('Gross Sales ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. Statistical Stationarity Verification

In [ ]:
# Run Augmented Dickey-Fuller (ADF) Test
def verify_stationarity(series):
    result = adfuller(series.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    if result[1] < 0.05:
        print("Conclusion: Series is stationary (Reject H0)")
    else:
        print("Conclusion: Series is non-stationary (Fail to reject H0). Transformation required.")

print("--- Baseline Check ---")
verify_stationarity(df['Sales'])

print("\n--- First-Order Differencing Check ---")
verify_stationarity(df['Sales'].diff())

## 3. Structural Decomposition Analysis

In [ ]:
decomposition = seasonal_decompose(df['Sales'], model='additive', period=12)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.show()

## 4. Algorithmic Modeling (SARIMAX Engine)

In [ ]:
# Split into Train/Validation groups (Last 6 months as validation test holdout)
train_data = df.iloc[:-6]
test_data = df.iloc[-6:]

# Fit optimal SARIMAX structural architecture
model = SARIMAX(train_data['Sales'], order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
results = model.fit(disp=False)
print(results.summary())

In [ ]:
# Generate out-of-sample forward projections
predictions = results.get_forecast(steps=6)
pred_df = predictions.predicted_mean
pred_df.index = test_data.index

# Calculate evaluation metrics
mape = mean_absolute_percentage_error(test_data['Sales'], pred_df)
rmse = root_mean_squared_error(test_data['Sales'], pred_df)
print(f"Validation Evaluation Matrix:")
print(f"Mean Absolute Percentage Error (MAPE): {mape*100:.2f}%")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")

In [ ]:
# Plot model forecast vs actual values
plt.figure(figsize=(12, 5))
plt.plot(train_data.index, train_data['Sales'], label='Historical Actuals', color='#1e293b')
plt.plot(test_data.index, test_data['Sales'], label='Validation Actuals', color='#2563eb', marker='o')
plt.plot(test_data.index, pred_df, label='Model Forecast Fit', color='#ef4444', linestyle='--', marker='x')
plt.title('Forecasting Engine Validation Performance Assessment')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()